In [46]:

class Article:
    def __init__(self, title, abstract, url):
        self.title = title
        self.abstract = abstract
        self.url = url

class Issue:
    def __init__(self, number, url):
        self.number = number
        self.url = url
        self.articles = []

class Volume:
    def __init__(self, number):
        self.number = number
        self.issues = []

In [44]:
import nltk
from collections import defaultdict, Counter

class NGramPredictor:
    def __init__(self, tokens):
        self.tokens = tokens
        self.unigrams = Counter(tokens)
        self.bigrams = Counter(nltk.ngrams(tokens, 2))
        self.trigrams = Counter(nltk.ngrams(tokens, 3))

    def prob_bigram(self, w1, w2):
        """P(w2 | w1)"""
        return self.bigrams[(w1, w2)] / self.unigrams[w1] if self.unigrams[w1] > 0 else 0

    def prob_trigram(self, w1, w2, w3):
        """P(w3 | w1, w2)"""
        return self.trigrams[(w1, w2, w3)] / self.bigrams[(w1, w2)] if self.bigrams[(w1, w2)] > 0 else 0

    def predict_next_word(self, sequence, λ1=0.4, λ2=0.6):
        tokens = nltk.word_tokenize(sequence.lower())
        
        if len(tokens) == 0:
            return None
        
        if len(tokens) == 1:
            w1 = tokens[-1]
            candidates = {w2: self.prob_bigram(w1, w2) for (_, w2) in self.bigrams if _ == w1}
            return max(candidates, key=candidates.get, default=None)
        
        else:
            w1, w2 = tokens[-2], tokens[-1]
            vocabulary = set(self.unigrams.keys())
            probs = {}
            for w3 in vocabulary:
                p_bigram = self.prob_bigram(w2, w3)
                p_trigram = self.prob_trigram(w1, w2, w3)
                p_interp = λ1 * p_bigram + λ2 * p_trigram
                probs[w3] = p_interp
            return max(probs, key=probs.get, default=None)


In [47]:
# apply model for tokens of an article
with open("data/volumes_with_tokens.pkl", "rb") as f:
    volumes = pickle.load(f)
first_article = volumes[0].issues[0].articles[0]
article_tokens = first_article.tokens  # assuming first_article is defined

In [49]:
model = NGramPredictor(article_tokens)
print("Next word after 'particle':", model.predict_next_word("particle"))
print("Next word after 'particle swarm':", model.predict_next_word("particle swarm"))


Next word after 'particle': None
Next word after 'particle swarm': tool


# Interface

# Interface

In [ ]:
import tkinter as tk
from tkinter import ttk, scrolledtext
from collections import Counter
import nltk
import pickle
import itertools
import math
import nltk
from collections import Counter


# Ensure you have the tokenizer downloaded
nltk.download('punkt', quiet=True)


with open("data/volumes_with_tokens.pkl", "rb") as f:
    volumes = pickle.load(f)

example_tokens =volumes[0].issues[0].articles[0].tokens

# --- NGram Predictor ---
class NGramPredictor:
    def __init__(self, tokens):
        self.tokens = tokens
        self.unigrams = Counter(tokens)
        self.bigrams = Counter(nltk.ngrams(tokens, 2))
        self.trigrams = Counter(nltk.ngrams(tokens, 3))

    def prob_bigram(self, w1, w2):
        return self.bigrams[(w1, w2)] / self.unigrams[w1] if self.unigrams[w1] > 0 else 0

    def prob_trigram(self, w1, w2, w3):
        return self.trigrams[(w1, w2, w3)] / self.bigrams[(w1, w2)] if self.bigrams[(w1, w2)] > 0 else 0

    def predict_next_word(self, sequence, λ1=0.4, λ2=0.6):
        tokens = nltk.word_tokenize(sequence.lower())
        if not tokens:
            return []

        # Case 1: Only one word → use bigram
        if len(tokens) == 1:
            w1 = tokens[-1]
            candidates = {w2: self.prob_bigram(w1, w2) for (a, w2) in self.bigrams if a == w1}
            return sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:5]

        # Case 2: Two or more → interpolate bigram and trigram
        w1, w2 = tokens[-2], tokens[-1]
        vocabulary = set(self.unigrams.keys())
        probs = {}
        for w3 in vocabulary:
            p_bigram = self.prob_bigram(w2, w3)
            p_trigram = self.prob_trigram(w1, w2, w3)
            p_interp = λ1 * p_bigram + λ2 * p_trigram
            if p_interp > 0:
                probs[w3] = p_interp
        return sorted(probs.items(), key=lambda x: x[1], reverse=True)[:5]

    def sequence_probability(self, sequence):
        """Compute probability of entire sequence using Bigram model"""
        tokens = nltk.word_tokenize(sequence.lower())
        if len(tokens) < 1:
            return 0
        prob = self.prob_unigram(tokens[0])  # start with unigram prob of first word
        for i in range(1, len(tokens)):
            p = self.prob_bigram(tokens[i - 1], tokens[i])
            prob *= p if p > 0 else 1e-8  # avoid zero
        return prob

    def compare_sequences(self, word_options):
        """
        word_options: list of lists -> each inner list = set of possible words for that position
        e.g. [["particle"], ["swarm", "optimization"], ["algorithm", "system"]]
        """
        all_combinations = list(itertools.product(*word_options))
        results = []
        for combo in all_combinations:
            seq = " ".join(combo)
            p = self.sequence_probability(seq)
            results.append((seq, p))
        return sorted(results, key=lambda x: x[1], reverse=True)
    

# --- Main Application ---
class NGramApp:
    def __init__(self, root):
        self.root = root
        self.root.title("🧠 N-Gram Next-Word Prediction Interface")
        self.root.geometry("950x700")

        # Scrollable container
        main_frame = tk.Frame(root)
        main_frame.pack(fill=tk.BOTH, expand=1)
        canvas = tk.Canvas(main_frame)
        scrollbar = ttk.Scrollbar(main_frame, orient=tk.VERTICAL, command=canvas.yview)
        scroll_frame = tk.Frame(canvas)

        scroll_frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=scroll_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)

        canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=1)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

        # Title
        ttk.Label(scroll_frame, text="🔮 Real-Time Next Word Prediction", font=("Arial", 16, "bold")).pack(anchor="w", pady=10)

        # Entry field
        self.input_text = tk.Entry(scroll_frame, width=70, font=("Arial", 14))
        self.input_text.pack(pady=10)
        self.input_text.bind("<KeyRelease>", self.on_typing)  # update on typing

        # Suggestion box
        self.suggest_box = tk.Listbox(scroll_frame, width=70, height=6, font=("Arial", 12))
        self.suggest_box.pack(pady=10)

        # Label for probability info
        self.info_label = ttk.Label(scroll_frame, text="", font=("Arial", 11))
        self.info_label.pack(pady=5)

        # Example corpus (replace with your real article tokens)
        self.tokens = example_tokens
        self.model = NGramPredictor(self.tokens)

        # Instruction
        ttk.Label(
            scroll_frame,
            text="💡 Type a few words (e.g., 'particle swarm') to see next-word suggestions appear below.",
            font=("Arial", 11, "italic")
        ).pack(anchor="w", pady=10)

    # --- Triggered on typing ---
    def on_typing(self, event=None):
        seq = self.input_text.get().strip()
        self.suggest_box.delete(0, tk.END)
        self.info_label.config(text="")

        if not seq:
            return

        suggestions = self.model.predict_next_word(seq)
        if suggestions:
            for word, prob in suggestions:
                self.suggest_box.insert(tk.END, f"{word:<15}  |  P = {prob:.5f}")
            self.info_label.config(text=f"Top {len(suggestions)} suggestions (sorted by highest probability)")
        else:
            self.suggest_box.insert(tk.END, "No suggestion available.")
        
    # --- PART 5: Entire Sequence Estimation ---
    ttk.Label(scroll_frame, text="🧩 Estimate Entire Sequence Probability", font=("Arial", 14, "bold")).pack(anchor="w", pady=10)

    self.sequence_input = tk.Entry(scroll_frame, width=70, font=("Arial", 12))
    self.sequence_input.pack(pady=5)

    ttk.Button(scroll_frame, text="Compute Sequence Probability", command=self.compute_sequence).pack(pady=5)

    self.sequence_display = tk.Text(scroll_frame, width=100, height=8, font=("Consolas", 11))
    self.sequence_display.pack(pady=5)

    def compute_sequence(self):
        seq = self.sequence_input.get().strip()
        if not seq:
            return
        p = self.model.sequence_probability(seq)
        self.sequence_display.delete("1.0", tk.END)
        self.sequence_display.insert(tk.END, f"Sequence: {seq}\n")
        self.sequence_display.insert(tk.END, f"Probability (Bigram Model): {p:.8e}\n")


# --- Run the App ---
if __name__ == "__main__":
    root = tk.Tk()
    app = NGramApp(root)
    root.mainloop()


In [58]:
import tkinter as tk
from tkinter import ttk
from collections import Counter
import nltk
import pickle
import itertools

# Ensure you have the tokenizer downloaded
nltk.download('punkt', quiet=True)

# Load your data
with open("data/volumes_with_tokens.pkl", "rb") as f:
    volumes = pickle.load(f)

example_tokens = volumes[0].issues[0].articles[0].tokens


# --- NGram Predictor ---
class NGramPredictor:
    def __init__(self, tokens):
        self.tokens = tokens
        self.unigrams = Counter(tokens)
        self.bigrams = Counter(nltk.ngrams(tokens, 2))
        self.trigrams = Counter(nltk.ngrams(tokens, 3))

    def prob_unigram(self, w):
        return self.unigrams[w] / sum(self.unigrams.values()) if self.unigrams[w] > 0 else 0

    def prob_bigram(self, w1, w2):
        return self.bigrams[(w1, w2)] / self.unigrams[w1] if self.unigrams[w1] > 0 else 0

    def prob_trigram(self, w1, w2, w3):
        return self.trigrams[(w1, w2, w3)] / self.bigrams[(w1, w2)] if self.bigrams[(w1, w2)] > 0 else 0

    def predict_next_word(self, sequence, λ1=0.4, λ2=0.6):
        tokens = nltk.word_tokenize(sequence.lower())
        if not tokens:
            return []

        # Case 1: One word → Bigram model
        if len(tokens) == 1:
            w1 = tokens[-1]
            candidates = {w2: self.prob_bigram(w1, w2) for (a, w2) in self.bigrams if a == w1}
            return sorted(candidates.items(), key=lambda x: x[1], reverse=True)[:5]

        # Case 2: ≥ 2 words → interpolation between Bigram and Trigram
        w1, w2 = tokens[-2], tokens[-1]
        vocabulary = set(self.unigrams.keys())
        probs = {}
        for w3 in vocabulary:
            p_bigram = self.prob_bigram(w2, w3)
            p_trigram = self.prob_trigram(w1, w2, w3)
            p_interp = λ1 * p_bigram + λ2 * p_trigram
            if p_interp > 0:
                probs[w3] = p_interp
        return sorted(probs.items(), key=lambda x: x[1], reverse=True)[:5]

    def sequence_probability(self, sequence):
        """Compute probability of an entire sequence using Bigram model"""
        tokens = nltk.word_tokenize(sequence.lower())
        if len(tokens) < 1:
            return 0
        prob = self.prob_unigram(tokens[0])
        for i in range(1, len(tokens)):
            p = self.prob_bigram(tokens[i - 1], tokens[i])
            prob *= p if p > 0 else 1e-8  # Avoid zero
        return prob

    def compare_sequences(self, word_options):
        """Generate all combinations and rank them by probability"""
        all_combinations = list(itertools.product(*word_options))
        results = []
        for combo in all_combinations:
            seq = " ".join(combo)
            p = self.sequence_probability(seq)
            results.append((seq, p))
        return sorted(results, key=lambda x: x[1], reverse=True)


# --- Main Application ---
class NGramApp:
    def __init__(self, root):
        self.root = root
        self.root.title("🧠 N-Gram Language Model Interface")
        self.root.geometry("950x750")

        # Scrollable main container
        main_frame = tk.Frame(root)
        main_frame.pack(fill=tk.BOTH, expand=1)
        canvas = tk.Canvas(main_frame)
        scrollbar = ttk.Scrollbar(main_frame, orient=tk.VERTICAL, command=canvas.yview)
        scroll_frame = tk.Frame(canvas)

        scroll_frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
        canvas.create_window((0, 0), window=scroll_frame, anchor="nw")
        canvas.configure(yscrollcommand=scrollbar.set)
        canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=1)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

        # --- PART 1: Next Word Prediction ---
        ttk.Label(scroll_frame, text="🔮 Real-Time Next Word Prediction", font=("Arial", 16, "bold")).pack(anchor="w", pady=10)

        self.input_text = tk.Entry(scroll_frame, width=70, font=("Arial", 14))
        self.input_text.pack(pady=10)
        self.input_text.bind("<KeyRelease>", self.on_typing)

        self.suggest_box = tk.Listbox(scroll_frame, width=70, height=6, font=("Arial", 12))
        self.suggest_box.pack(pady=10)

        self.info_label = ttk.Label(scroll_frame, text="", font=("Arial", 11))
        self.info_label.pack(pady=5)

        self.tokens = example_tokens
        self.model = NGramPredictor(self.tokens)

        ttk.Label(
            scroll_frame,
            text="💡 Type a few words (e.g., 'particle swarm') to see next-word suggestions below.",
            font=("Arial", 11, "italic")
        ).pack(anchor="w", pady=10)

        # --- PART 2: Entire Sequence Probability ---
        ttk.Label(scroll_frame, text="🧩 Estimate Entire Sequence Probability", font=("Arial", 14, "bold")).pack(anchor="w", pady=10)

        self.sequence_input = tk.Entry(scroll_frame, width=70, font=("Arial", 12))
        self.sequence_input.pack(pady=5)

        ttk.Button(scroll_frame, text="Compute Sequence Probability", command=self.compute_sequence).pack(pady=5)

        self.sequence_display = tk.Text(scroll_frame, width=100, height=8, font=("Consolas", 11))
        self.sequence_display.pack(pady=5)

    # --- Typing Event ---
    def on_typing(self, event=None):
        seq = self.input_text.get().strip()
        self.suggest_box.delete(0, tk.END)
        self.info_label.config(text="")

        if not seq:
            return

        suggestions = self.model.predict_next_word(seq)
        if suggestions:
            for word, prob in suggestions:
                self.suggest_box.insert(tk.END, f"{word:<15} | P = {prob:.5f}")
            self.info_label.config(text=f"Top {len(suggestions)} suggestions (sorted by highest probability)")
        else:
            self.suggest_box.insert(tk.END, "No suggestion available.")

    # --- Sequence Probability Computation ---
    def compute_sequence(self):
        seq = self.sequence_input.get().strip()
        if not seq:
            return
        p = self.model.sequence_probability(seq)
        self.sequence_display.delete("1.0", tk.END)
        self.sequence_display.insert(tk.END, f"Sequence: {seq}\n")
        self.sequence_display.insert(tk.END, f"Probability (Bigram Model): {p:.8e}\n")


# --- Run the App ---
if __name__ == "__main__":
    root = tk.Tk()
    app = NGramApp(root)
    root.mainloop()


# Part 2